# EMRI WDM TDI Benchmark

Benchmark notebook for dense WDM coefficient EMRI response generation and Gen-2 TDI assembly, focused on end-to-end runtime and accuracy.

Dense CPU optimization phase closure note:
- Dense CPU WDM baseline was validated and benchmarked.
- assembly_vmap=True provided the primary useful dense CPU gain.
- Additional dense optimization branches were tested and rejected (no material net runtime improvement).
- The dense reference method is frozen here for future comparisons against more structural/sparse approaches.

In [ ]:
import os
import sys
import time
from pathlib import Path

# CPU-only optimization pass: force JAX CPU backend before importing jax.
os.environ['JAX_PLATFORM_NAME'] = 'cpu'

import numpy as np
import pandas as pd

import jax
jax.config.update('jax_enable_x64', True)
jax.config.update('jax_platform_name', 'cpu')

ROOT = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in [ROOT, *ROOT.parents] if (p / 'WDM_LISAresponse').exists() and (p / 'WDM_GW_wavelets').exists()),
    ROOT,
 )
WDM_LISA_ROOT = PROJECT_ROOT / 'WDM_LISAresponse'
WDM_GW_ROOT = PROJECT_ROOT / 'WDM_GW_wavelets'
WDM_LISA_SRC = WDM_LISA_ROOT / 'src'
WDM_SRC = WDM_GW_ROOT / 'src'

for p in [str(WDM_LISA_SRC), str(WDM_SRC), str(PROJECT_ROOT)]:
    if p not in sys.path:
        sys.path.insert(0, p)

import WDM
from lisaorbits.file import read_orbit_file
from lisaorbits import InterpolatedOrbits
from few.waveform import GenerateEMRIWaveform
from few.utils.constants import YRSID_SI

from WDM_LISAresponse.code.lisa_response import response as response_mod
from WDM_LISAresponse.code.lisa_response.waveforms import generate_windowed_padded_waveform
from WDM_LISAresponse.code.lisa_response.TDI import (
    gen2_sagnac_wdm,
    gen2_michelson_wdm,
    gen2_michelson_td,
 )
from WDM_LISAresponse.code.utils.utils import compute_wdm_cut_info, apply_wdm_cut, wdm_mismatch

In [ ]:
import importlib
import WDM.code.time_delay_filters._time_shift_jax as _ts_jax_reload
import WDM.code.time_delay_filters._time_shift_assembly as _ts_asm_reload
import WDM.code.time_delay_filters.time_shift_fast as _ts_fast_reload
from WDM_LISAresponse.code.lisa_response import response_sub as _response_sub_reload
from WDM_LISAresponse.code.lisa_response import response as _response_reload
from WDM_LISAresponse.code.lisa_response import TDI as _tdi_reload

importlib.reload(_ts_jax_reload)
importlib.reload(_ts_asm_reload)
importlib.reload(_ts_fast_reload)
importlib.reload(_response_sub_reload)
importlib.reload(_response_reload)
importlib.reload(_tdi_reload)

response_mod = _response_reload
gen2_sagnac_wdm = _tdi_reload.gen2_sagnac_wdm
gen2_michelson_wdm = _tdi_reload.gen2_michelson_wdm

In [ ]:
BASELINE = {
    'dt': 10.0,
    'Nf': 2000,
    'L_trunc': 25,
    'q': 4,
    'calc_m0': True,
    'd': 4,
    'tl_tp_mode': 'interp',
    'tl_tp_interp_points': 2,
    'tl_tp_interp_kind': 'linear',
    'tl_tp_interp_pad': 0.0,
    'assembly_vmap': False,
    'batch_chunk': 32,
    'alpha': 0.05,
    'tukey_factor': 1.0,
}

T_OBS_YEARS = 0.25
N_SAMPLES = int(np.floor(T_OBS_YEARS * YRSID_SI / BASELINE['dt']))

EMRI_PARAMS = {
    'm1': 1e6, 'm2': 10.0, 'a': 0.9, 'p0': 8.5, 'e0': 0.3, 'x0': 1.0,
    'dist': 1.0, 'theta': np.pi / 2.0, 'phi': 0.0,
    'qK': np.pi / 3.0, 'phiK': np.pi / 3.0,
    'Phi_phi0': np.pi / 3.0, 'Phi_theta0': 0.0, 'Phi_r0': np.pi / 3.0,
    'eps': 1e-2,
}

In [ ]:
td_gen = GenerateEMRIWaveform(
    'FastKerrEccentricEquatorialFlux',
    sum_kwargs=dict(pad_output=True, odd_len=True),
    return_list=True,
)

orbit_data = read_orbit_file(str(WDM_LISA_ROOT / 'examples' / 'esa-trailing-orbits.h5'))
orbit_times = orbit_data.t0 + orbit_data.dt * np.arange(orbit_data.size)
orbits = InterpolatedOrbits(orbit_times, orbit_data.positions, orbit_data.velocities)
t0 = float(orbit_data.t0 + 1.0 * YRSID_SI)

In [ ]:
def build_shift_kwargs(cfg):
    mode = str(cfg.get('tl_tp_mode', 'interp')).lower()
    if mode == 'exact':
        raise ValueError("tl_tp_mode='exact' is disabled in this notebook for memory-safe runs.")
    return {
        'use_jax': True,
        'tl_tp_mode': 'interp',
        'tl_tp_interp_points': int(cfg['tl_tp_interp_points']),
        'tl_tp_interp_kind': cfg['tl_tp_interp_kind'],
        'tl_tp_interp_pad': float(cfg['tl_tp_interp_pad']),
        'batch_chunk': cfg.get('batch_chunk', 32),
        'assembly_vmap': bool(cfg.get('assembly_vmap', True)),
    }

def generate_emri(cfg, n_samples):
    model_kwargs = dict(EMRI_PARAMS)
    model_kwargs['td_gen'] = td_gen
    return generate_windowed_padded_waveform(
        waveform_model='emri',
        t0=t0,
        dt=float(cfg['dt']),
        N=int(n_samples),
        model_kwargs=model_kwargs,
        window_function='tukey',
        window_kwargs={'alpha': cfg['alpha'], 'tukey_factor': cfg['tukey_factor']},
        wdm_Nf=int(cfg['Nf']),
        return_padding_info=True,
    )

def prebuild_waveform_payloads(cfg, sizes):
    payloads = {}
    for n in sizes:
        n_i = int(n)
        if n_i <= 0:
            continue
        payloads[n_i] = generate_emri(cfg, n_i)
    return payloads

In [ ]:
def timed_ms(fn):
    t0_local = time.perf_counter()
    out = fn()
    return out, (time.perf_counter() - t0_local) * 1000.0

def run_pipeline(cfg, n_samples, waveform_payload=None, include_sagnac=False):
    stages = {}
    if waveform_payload is None:
        (times_pad, hp, hc, pad_info), stages['waveform_ms'] = timed_ms(lambda: generate_emri(cfg, n_samples))
    else:
        times_pad, hp, hc, pad_info = waveform_payload
        stages['waveform_ms'] = 0.0

    shift_kwargs = build_shift_kwargs(cfg)
    one_arm_cache = {}
    shifts_cache = {}

    (wdm_obj, _), stages['one_arm_ms'] = timed_ms(
        lambda: response_mod.precompute_one_arm_wdm_dT_links(
            times_pad, orbits, EMRI_PARAMS['theta'], EMRI_PARAMS['phi'], hp, hc,
            Nf=int(cfg['Nf']), d=int(cfg['d']), L_trunc=int(cfg['L_trunc']),
            cache=one_arm_cache, variable_shift_kwargs=shift_kwargs,
            use_jax=True, q=int(cfg['q']), calc_m0=bool(cfg['calc_m0']),
        )
    )

    if include_sagnac:
        (_, _, _), stages['sagnac_ms'] = timed_ms(
            lambda: gen2_sagnac_wdm(
                times_pad, hp, hc, EMRI_PARAMS['theta'], EMRI_PARAMS['phi'], orbits,
                Nf=int(cfg['Nf']), shifts_cache=shifts_cache, L_trunc=int(cfg['L_trunc']),
                one_arm_cache=one_arm_cache, variable_shift_kwargs=shift_kwargs,
                use_jax=True, q=int(cfg['q']),
            )
        )
    else:
        stages['sagnac_ms'] = 0.0

    (X1, Y1, Z1), stages['michelson_ms'] = timed_ms(
        lambda: gen2_michelson_wdm(
            times_pad, hp, hc, EMRI_PARAMS['theta'], EMRI_PARAMS['phi'], orbits,
            Nf=int(cfg['Nf']), shifts_cache=shifts_cache, L_trunc=int(cfg['L_trunc']),
            one_arm_cache=one_arm_cache, variable_shift_kwargs=shift_kwargs,
            use_jax=True, q=int(cfg['q']),
        )
    )

    stages['total_ms'] = stages['waveform_ms'] + stages['one_arm_ms'] + stages['sagnac_ms'] + stages['michelson_ms']
    return {
        'stages': stages,
        'times_pad': times_pad,
        'hplus': hp,
        'hcross': hc,
        'pad_info': pad_info,
        'wdm_obj': wdm_obj,
        'X1': X1, 'Y1': Y1, 'Z1': Z1,
    }

In [ ]:
BASE_N_SAMPLES = N_SAMPLES
BASE_CFG = dict(BASELINE)
BASE_CFG['assembly_vmap'] = True
BASE_CFG['calc_m0'] = True

WAVEFORMS, waveform_build_ms = timed_ms(
    lambda: prebuild_waveform_payloads(BASE_CFG, [BASE_N_SAMPLES])
)
WAVEFORM_BUILD_MS = float(waveform_build_ms)

times_pad_ref, hp_ref, hc_ref, pad_info_ref = WAVEFORMS[BASE_N_SAMPLES]
BLANK_WAVEFORMS = {
    BASE_N_SAMPLES: (
        times_pad_ref,
        np.zeros_like(hp_ref),
        np.zeros_like(hc_ref),
        pad_info_ref,
    )
}

def _run_baseline(cfg):
    # Warmup pass for stable dense baseline timing.
    _ = run_pipeline(cfg, BASE_N_SAMPLES, waveform_payload=BLANK_WAVEFORMS[BASE_N_SAMPLES], include_sagnac=False)
    _ = run_pipeline(cfg, BASE_N_SAMPLES, waveform_payload=WAVEFORMS[BASE_N_SAMPLES], include_sagnac=False)
    return run_pipeline(cfg, BASE_N_SAMPLES, waveform_payload=WAVEFORMS[BASE_N_SAMPLES], include_sagnac=False)

baseline_run = _run_baseline(BASE_CFG)
baseline_total = float(baseline_run['stages']['total_ms'])

dense_optimization_table = pd.DataFrame([
    {
        'case_name': 'current_vmap_wdm_baseline',
        'runtime_one_arm_ms': float(baseline_run['stages']['one_arm_ms']),
        'runtime_xyz_ms': float(baseline_run['stages']['michelson_ms']),
        'runtime_total_ms': baseline_total,
        'speedup_vs_current_vmap_baseline': 1.0,
        'overlap_vs_current_vmap_baseline': 1.0,
        'mismatch_vs_current_vmap_baseline': 0.0,
        'notes': 'Dense baseline frozen after optimization phase closure.',
    }
])[[
    'case_name',
    'runtime_one_arm_ms',
    'runtime_xyz_ms',
    'runtime_total_ms',
    'speedup_vs_current_vmap_baseline',
    'overlap_vs_current_vmap_baseline',
    'mismatch_vs_current_vmap_baseline',
    'notes',
]].round(6)

wdm_run_timing_table = dense_optimization_table
wdm_vmap_comparison_table = dense_optimization_table[[
    'case_name',
    'overlap_vs_current_vmap_baseline',
    'mismatch_vs_current_vmap_baseline',
]].copy()

dense_optimization_table

In [ ]:
# CPU-only optimization summary views reused by display cells below.
one_arm_table = dense_optimization_table[[
    'case_name', 'runtime_one_arm_ms', 'speedup_vs_current_vmap_baseline', 'notes'
]].copy()

michelson_table = dense_optimization_table[[
    'case_name', 'runtime_xyz_ms', 'speedup_vs_current_vmap_baseline', 'notes'
]].copy()

timing_table = dense_optimization_table[[
    'case_name',
    'runtime_one_arm_ms',
    'runtime_xyz_ms',
    'runtime_total_ms',
    'speedup_vs_current_vmap_baseline',
]].copy()

timing_stage_table = pd.DataFrame([
    {'stage': 'runtime_one_arm_ms'},
    {'stage': 'runtime_xyz_ms'},
    {'stage': 'runtime_total_ms'},
])

one_arm_table

In [ ]:
michelson_table

In [ ]:
wdm_run_timing_table

In [ ]:
timing_table

In [ ]:
wdm_vmap_comparison_table

In [ ]:
timing_stage_table